# NB14 — LOCO Median Recompute

**Purpose:** Address the critique that the reported LOCO (Leave-One-Country-Out) mean
AUROC=0.893 for EBM is inflated by countries with only a single crisis event in their
held-out fold (e.g. BEL, DNK, ESP, FIN), where AUROC=1.000 is mechanically guaranteed
regardless of model quality (a single positive case can always be ranked above the
negatives). The dissertation already notes this in passing but still reports the
unadjusted mean as if it were a clean summary statistic.

**This notebook does not retrain anything.** It reads NB08B's already-saved
`nb08b_loco_validation.csv` (which includes an `n_holdout` column — the number of crisis
events in each country's held-out fold) and recomputes the summary statistics the correct
way: median AUROC, and mean AUROC excluding single-event countries.

**Input:** `nb08b_loco_validation.csv` (from NB08B) — read-only, no modification.


## Cell 1 — Load NB08B's saved LOCO results

In [ ]:
import pandas as pd
from pathlib import Path

BASE    = Path(r'.')
ROB_DIR = BASE / 'data' / 'processed' / 'augmented_robustness'

loco_path = ROB_DIR / 'nb08b_loco_validation.csv'
print(f'Loading : {loco_path}')
print(f'Exists  : {loco_path.exists()}')

loco_df = pd.read_csv(loco_path)
print(f'Rows loaded: {len(loco_df)}')
print(f'Models present: {sorted(loco_df["model"].unique())}')


## Cell 2 — Full per-country breakdown (EBM M2.5), sorted by hold-out event count

In [ ]:
ebm = loco_df[loco_df['model'] == 'EBM M2.5'].copy()
rf  = loco_df[loco_df['model'] == 'M2 RF'].copy()

print('=== LOCO — FULL BREAKDOWN (EBM M2.5), sorted by n_holdout ===')
print(ebm[['country', 'n_holdout', 'AUROC', 'AUPRC']].sort_values('n_holdout').to_string(index=False))
print()

single_event = ebm[ebm['n_holdout'] == 1]
multi_event  = ebm[ebm['n_holdout'] > 1]

print(f'Single-event countries (n_holdout=1, n={len(single_event)}): {list(single_event["country"])}')
print(f'  -> AUROC values: {list(single_event["AUROC"].round(3))}')
print()
print(f'Multi-event countries (n_holdout>1, n={len(multi_event)}): {list(multi_event["country"])}')


## Cell 3 — Corrected summary statistics: EBM vs RF

In [ ]:
print('=' * 90)
print(' LOCO SUMMARY STATISTICS — ORIGINAL vs CORRECTED')
print('=' * 90)
print()

summary_rows = []
for label, df in [('EBM M2.5', ebm), ('M2 RF', rf)]:
    full_mean   = df['AUROC'].mean()
    full_std    = df['AUROC'].std()
    full_median = df['AUROC'].median()
    n_single    = int((df['n_holdout'] == 1).sum())
    multi_only  = df[df['n_holdout'] > 1]['AUROC']
    excl_mean   = multi_only.mean()
    excl_std    = multi_only.std()

    print(f'--- {label} ---')
    print(f'  Original reported (mean, all {len(df)} countries)              : {full_mean:.4f}  (std={full_std:.4f})')
    print(f'  Median (all {len(df)} countries)                               : {full_median:.4f}')
    print(f'  Mean EXCLUDING single-event countries ({len(multi_only)} countries)      : {excl_mean:.4f}  (std={excl_std:.4f})')
    print(f'  Single-event countries excluded (n={n_single})                        : {list(df[df["n_holdout"]==1]["country"])}')
    print()

    summary_rows.append({
        'Model': label,
        'Original mean (all)': round(full_mean, 4),
        'Median (all)': round(full_median, 4),
        'Mean (excl. single-event)': round(excl_mean, 4),
        'N single-event excluded': n_single,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

summary_df.to_csv(ROB_DIR / 'nb14_loco_corrected_summary.csv', index=False)
print(f'\nSaved -> {ROB_DIR / "nb14_loco_corrected_summary.csv"}')


## Cell 4 — Completion summary

In [ ]:
print('=' * 65)
print(' NB14 -- LOCO MEDIAN RECOMPUTE COMPLETE')
print('=' * 65)
print()
print('No existing files (NB08B outputs) were modified.')
print('Next: paste the printed summary table back to Claude for review before')
print('updating Ch4/Table wording from mean to median / mean-excl-single-event.')
